假设客服主管每天都要判断哪些工单应该优先处理。工作一段时间后，他可能发现：

- VIP 客户的紧急工单比较多；
- 深夜提交并且描述较长的工单，可能更紧急；
- 提供了订单号的工单，通常更容易快速确认问题；
- 只看某一个条件，经常会判断错误。

In [55]:
# 准备一条新工单的信息
customer_level = "vip"
submit_hour = 23
message_length = 180
has_order_id = 1

# 按人工编写的固定条件判断优先级
if customer_level == "vip" and submit_hour >= 22:
    priority = "高优先级"
elif message_length >= 150 and has_order_id == 1:
    priority = "高优先级"
else:
    priority = "普通"

# 输出规则判断结果
print(priority)

高优先级


机器学习主要完成的是：

过去的数据和结果 → 学习规律 → 对新数据进行预测或分组

### 2.2 样本、特征与标签

#### 1. 样本：一个被学习或预测的对象

```plain
一条工单       → 一个样本
一位客户       → 一个样本
一套房屋       → 一个样本
一张商品图片    → 一个样本
```


#### 2. 特征：模型进行判断时使用的信息

- 提交渠道；
- 客户等级；
- 是否提供订单号；
- 提交小时；
- 描述长度。

#### 3. 标签：训练时提供的历史答案

```plain
工单优先级分类  → 标签是“高优先级”或“普通”
处理时长预测    → 标签是实际处理分钟数
房屋价格预测    → 标签是实际成交价格
```

标签主要用于监督学习。分类和回归通常需要标签；聚类等无监督学习任务通常没有预先提供的标签。

标签并不一定是绝对正确的“真理”。如果历史记录错误、人工标准不统一，模型也会学习到错误或互相矛盾的规律。

### 2.4 算法、训练、模型与预测

```plain
逻辑回归                     → 一种学习算法
历史工单                     → 训练数据
使用历史工单调整内部规律        → 训练过程
训练完成的工单分类器           → 具体模型
使用分类器判断一条新工单        → 预测
```

### 2.5 演示：在代码中拆分特征 X 和标签 y

In [56]:
import pandas as pd

# 1. 准备 4 条带标签的历史工单
sample_tickets = pd.DataFrame(
    {
        "channel": ["web", "phone", "app", "email"],
        "customer_level": ["normal", "vip", "enterprise", "normal"],
        "has_order_id": [0, 1, 1, 1],
        "message_length": [35, 180, 210, 68],
        "is_high_priority": [0, 1, 1, 0],
    }
)

# 2. 指定模型可以使用的特征列
feature_columns_demo = [
    "channel",
    "customer_level",
    "has_order_id",
    "message_length",
]

# 3. X 保存输入特征，y 保存正确答案
X_demo = sample_tickets[feature_columns_demo]
y_demo = sample_tickets["is_high_priority"]

# 4. 观察特征和标签的内容与形状
#X.shape = (样本数量, 特征数量)
# y.shape = (样本数量,)
print("特征 X：")
print(X_demo)
print("X 的形状：", X_demo.shape)

print("\n标签 y：")
print(y_demo)
print("y 的形状：", y_demo.shape)

特征 X：
  channel customer_level  has_order_id  message_length
0     web         normal             0              35
1   phone            vip             1             180
2     app     enterprise             1             210
3   email         normal             1              68
X 的形状： (4, 4)

标签 y：
0    0
1    1
2    1
3    0
Name: is_high_priority, dtype: int64
y 的形状： (4,)


## 三、三类常见机器学习任务：分类、回归与聚类

### 3.1 从业务目标判断任务类型

分类、回归和聚类是模型需要完成的任务，监督学习、无监督学习等描述的是模型获得学习信号的方式，两组概念不在同一个层次。

1.分类:得到所属类别(高优先级/普通优先级)(二分类，多分类)

2.回归:预测一个可度量的数值(处理时间40min)

3.聚类:在没有标签时发现数据分组(第0组/第1组/第2组)

因此，"分类"和"回归"通常属于"监督学习"，"聚类"通常属于"无监督学习"。这里的“通常”表示常见对应关系，不代表所有机器学习问题都只能按照这一种方式处理。

| 对比维度       | 分类         | 回归           | 聚类            |
| ---------- | ---------- | ------------ | ------------- |
| 主要目标       | 判断样本属于哪个类别 | 估计一个连续数值     | 发现数据中的相似分组    |
| 是否需要已知标签   | 通常需要       | 通常需要         | 不需要预设标签       |
| 是否存在目标 `y` | 有，类别标签     | 有，数值标签       | 通常没有          |
| 模型输出       | 已知类别之一     | 数值预测结果       | 簇编号或分组结果      |
| 常见学习方式     | 监督学习       | 监督学习         | 无监督学习         |
| 工单案例       | 判断是否需要优先处理 | 预测处理时长       | 发现相似问题群组      |
| 结果如何使用     | 触发分类后的业务流程 | 为计划和决策提供数值参考 | 帮助探索数据并形成业务假设 |

## 四、认识 scikit-learn

scikit-learn 是 Python 中常用的机器学习工具库，提供了数据划分、数据预处理、传统机器学习算法和模型评估等功能。

它主要用于传统机器学习，特别适合处理表格数据中的分类、回归和聚类问题。它不是专门用于训练深度神经网络或大语言模型的框架。

| 导入位置                      | 主要作用           |
| ------------------------- | -------------- |
| `sklearn.model_selection` | 划分训练集、验证集和测试集  |
| `sklearn.preprocessing`   | 标准化和类别编码       |
| `sklearn.impute`          | 填充缺失值          |
| `sklearn.compose`         | 对不同列使用不同处理方式   |
| `sklearn.pipeline`        | 串联预处理和模型       |
| `sklearn.tree`            | 使用决策树分类或回归模型   |
| `sklearn.linear_model`    | 使用线性回归、逻辑回归等模型 |
| `sklearn.metrics`         | 计算模型评估指标       |

scikit-learn 中常见的对象可以先按职责分成两类：

| 对象角色   | 主要职责                           | 常见工具及作用                                                                                        |
| ------ | ------------------------------ | ---------------------------------------------------------------------------------------------- |
| 转换器    | 按照一定规则，把原始特征 `X` 变成处理后的新特征     | `StandardScaler`：调整数值尺度<br/>`OneHotEncoder`：把类别转换成数值特征                                         |
| 监督学习模型 | 学习特征 `X` 与标签 `y` 的关系，并对新样本给出预测 | `DecisionTreeClassifier`：预测类别<br/>`LinearRegression`：预测连续数值<br/>`LogisticRegression`：预测类别和类别概率 |

**记忆方法**：`fit` 表示学习，`transform` 表示转换特征，`predict` 表示给出预测。

In [57]:
import pandas as pd
from sklearn.linear_model import LinearRegression   #使用线性回归、逻辑回归等模型
from sklearn.tree import DecisionTreeClassifier     #使用决策树分类或回归模型

# 1. 准备包含输入字段和结果字段的完整历史工单数据
ticket_history = pd.DataFrame(
    {
        # 输入特征：客户已经等待多少分钟
        "waiting_minutes": [5, 15, 30, 60, 90, 120],
        # 输入特征：客户已经催单多少次
        "reminder_count": [0, 0, 1, 1, 2, 3],
        # 分类标签：0 表示普通，1 表示高优先级
        "priority": [0, 0, 0, 1, 1, 1],
        # 回归标签：工单实际处理分钟数
        "handle_minutes": [15, 22, 35, 50, 70, 90],
    }
)

# 2. 从完整数据中选出两个输入特征 X
basic_feature_columns = ["waiting_minutes", "reminder_count"]
X_tickets = ticket_history[basic_feature_columns]

# 3. 从完整数据中取得分类标签 y
y_priority = ticket_history["priority"]

# 4. 从完整数据中取得回归标签 y
y_handle_minutes = ticket_history["handle_minutes"]

# 5. 查看完整数据、输入特征和两个标签
print("完整历史工单数据：")
print(ticket_history)
print("输入特征 X：")
print(X_tickets)
print("分类标签 y_priority：")
print(y_priority)
print("回归标签 y_handle_minutes：")
print(y_handle_minutes)

# 6. 准备一条需要同时预测优先级和处理时长的新工单(特征X)
new_ticket = pd.DataFrame(
    {
        "waiting_minutes": [75],
        "reminder_count": [2],
    }
)

# 7. 创建并训练分类模型
ticket_classifier = DecisionTreeClassifier(
    # 限制决策树最多进行两层判断，避免树过于复杂
    max_depth=2,
    # 固定训练中的随机选择，使多次运行结果可以复现
    random_state=42,
)
ticket_classifier.fit(X_tickets, y_priority)

# 8. 预测新工单属于哪个优先级类别
predicted_priority = ticket_classifier.predict(new_ticket)[0]
priority_name = "高优先级" if predicted_priority == 1 else "普通"

# 9. 创建并训练回归模型
handle_time_regressor = LinearRegression()
handle_time_regressor.fit(X_tickets, y_handle_minutes)

# 10. 预测新工单的处理时长
predicted_minutes = handle_time_regressor.predict(new_ticket)[0]

# 11. 输出两个任务的预测结果
print("优先级预测：", priority_name)
print(f"预计处理时长：{predicted_minutes:.1f} 分钟")

完整历史工单数据：
   waiting_minutes  reminder_count  priority  handle_minutes
0                5               0         0              15
1               15               0         0              22
2               30               1         0              35
3               60               1         1              50
4               90               2         1              70
5              120               3         1              90
输入特征 X：
   waiting_minutes  reminder_count
0                5               0
1               15               0
2               30               1
3               60               1
4               90               2
5              120               3
分类标签 y_priority：
0    0
1    0
2    0
3    1
4    1
5    1
Name: priority, dtype: int64
回归标签 y_handle_minutes：
0    15
1    22
2    35
3    50
4    70
5    90
Name: handle_minutes, dtype: int64
优先级预测： 高优先级
预计处理时长：62.3 分钟


两类监督学习任务的基本调用顺序相同：

```plain
准备特征 X 和标签 y
→ 创建模型对象
→ 调用 fit(X, y) 训练
→ 调用 predict(X_new) 预测
```

## 五、训练集、验证集和测试集

| 数据集 | 主要作用                  | 考试类比 |
| --- | --------------------- | ---- |
| 训练集 | 通过 `fit` 学习预处理规则和模型规律 | 平时作业 |
| 验证集 | 比较不同特征、模型和训练配置        | 模拟考试 |
| 测试集 | 最终检查确定方案的实际效果         | 最终考试 |

| 参数             | 作用               | 本例                       |
| -------------- | ---------------- | ------------------------ |
| `test_size`    | 指定划出去的数据比例或数量    | 第一次划出 `0.4`，第二次划出临时数据的一半 |
| `random_state` | 固定随机划分结果，使实验可以复现 | 使用 `42`                  |
| `stratify`     | 尽量保持划分前后的类别比例    | 按工单优先级标签分层               |

In [58]:
import pandas as pd
from sklearn.model_selection import train_test_split

# 1. 构造 20 条只用于观察划分结果的示例数据
split_demo = pd.DataFrame({
    'ticket_id': [f'T{i:03d}' for i in range(1, 21)],
    'is_high_priority': [0, 1] * 10
})

# 2. 第一次划分：60% 作为训练集，40% 暂存
train_demo, temp_demo = train_test_split(
    split_demo,
    test_size=0.4,   #temp_demo能分到的份量
    random_state=42,
    stratify=split_demo['is_high_priority'], #标签不参与分割
)

# 3. 第二次划分：把 40% 的暂存数据平分
valid_demo, test_demo = train_test_split(
    temp_demo,
    test_size=0.5,
    random_state=42,
    stratify=temp_demo['is_high_priority'],
)

# 4. 检查每个集合的数量和标签分布
for dataset_name, dataset_frame in [
    ("训练集", train_demo),
    ("验证集", valid_demo),
    ("测试集", test_demo),
]:
    #集合数量（特征）
    print(f'\n{dataset_name}数量: {len(dataset_frame)}')
    #标签分布
    print(
        dataset_frame['is_high_priority']
        .value_counts()     #统计每个取值出现次数
        .sort_index()       #按索引排序,让 False 在前、True 在后（或按 0/1 排）
    )


训练集数量: 12
is_high_priority
0    6
1    6
Name: count, dtype: int64

验证集数量: 4
is_high_priority
0    2
1    2
Name: count, dtype: int64

测试集数量: 4
is_high_priority
0    2
1    2
Name: count, dtype: int64


#### 防止数据泄漏

**数据泄漏**是指：训练模型时，提前使用了验证集或测试集中的信息。

错误顺序:
```plain
全部数据
→ 使用全部数据计算缺失值填充值、均值、标准差或类别编码规则
→ 再划分训练集、验证集和测试集
→ 训练并评价模型
```

正确顺序:
```plain
先划分原始数据
→ 只使用训练集执行 fit 或 fit_transform
→ 使用训练集学到的规则 transform 验证集和测试集
→ 使用训练好的模型 predict 验证集和测试集
```

## 六、特征工程：把原始数据变成有效信息

### 6.1 特征工程的三个步骤

| 步骤   | 要解决的问题            | 工单案例                    |
| ---- | ----------------- | ----------------------- |
| 选择特征 | 哪些原始字段可以在预测时使用    | 保留渠道和客户等级，删除工单编号和最终处理时长 |
| 构造特征 | 怎样从原始字段中提取更有意义的信息 | 从提交时间提取小时，从问题描述中提取文本长度  |
| 转换特征 | 怎样把特征变成模型可以处理的形式  | 填充缺失值、标准化数值、编码字符串类别     |

数据清洗和特征工程经常连续进行，但关注点不同：

- 数据清洗关注数据是否正确、完整，例如处理重复值和错误格式；
- 特征工程关注哪些信息有助于预测，以及怎样把这些信息交给模型。

### 6.2 演示：从原始工单构造特征

In [59]:
import pandas as pd

# 1. 准备包含原始时间、文本和业务结果的工单数据
raw_tickets = pd.DataFrame(
    [
        {
            # 工单的唯一编号，只用于识别工单
            "ticket_id": "T101",
            # 工单提交时间
            "submitted_at": "2026-07-21 23:15:00",
            # 工单提交渠道
            "channel": "phone",
            # 客户等级
            "customer_level": "vip",
            # 是否提供订单号：1 表示是，0 表示否
            "has_order_id": 1,
            # 客户提交的问题描述
            "message": "订单一直没有收到，请尽快处理！",
            # 实际处理时长，单位为分钟
            "actual_handle_minutes": 18,
            # 是否为高优先级工单：1 表示是，0 表示否
            "is_high_priority": 1,
        },
        # 后续工单使用相同的字段结构
        {
            "ticket_id": "T102",
            "submitted_at": "2026-07-22 10:20:00",
            "channel": "web",
            "customer_level": "normal",
            # None 表示这条工单没有记录是否提供订单号
            "has_order_id": None,
            "message": "请问怎样修改收货地址？",
            "actual_handle_minutes": 35,
            "is_high_priority": 0,
        },
        {
            "ticket_id": "T103",
            "submitted_at": "2026-07-22 02:10:00",
            "channel": "app",
            # None 表示这条工单没有记录客户等级
            "customer_level": None,
            "has_order_id": 1,
            "message": "系统无法下单，业务已经中断！",
            "actual_handle_minutes": 12,
            "is_high_priority": 1,
        },
        {
            "ticket_id": "T104",
            "submitted_at": "2026-07-22 15:40:00",
            "channel": "email",
            "customer_level": "normal",
            "has_order_id": 1,
            "message": "想咨询发票开具流程。",
            "actual_handle_minutes": 42,
            "is_high_priority": 0,
        },
    ]
)

# 2. 将字符串时间转换为 pandas 日期时间类型
raw_tickets['submitted_at'] = pd.to_datetime(raw_tickets['submitted_at'])

# 3. 从提交时间中提取小时
raw_tickets["submit_hour"] = raw_tickets["submitted_at"].dt.hour

# 4. 根据小时构造是否深夜提交
raw_tickets['is_night'] = (
    (raw_tickets['submit_hour'] >= 22)
    | (raw_tickets['submit_hour'] < 6)
).astype(int)

# 5. 从工单文本构造长度和感叹号特征（.str.len()和.str.contains()遇到NaN结果是NaN）
raw_tickets['message_length'] = raw_tickets['message'].str.len()    #文本长度
raw_tickets['has_exclamation'] = (
    raw_tickets['message'].str.contains("!", regex=False).astype(int)
)   # regex=False：把 ! 当普通字符匹配

# 6. 查看构造后的特征
print(
    raw_tickets[
        [
            "ticket_id",
            "customer_level",
            "has_order_id",
            "submit_hour",
            "is_night",
            "message_length",
            "has_exclamation",
            "is_high_priority",
        ]
    ]
)

  ticket_id customer_level  has_order_id  submit_hour  is_night  \
0      T101            vip           1.0           23         1   
1      T102         normal           NaN           10         0   
2      T103            NaN           1.0            2         1   
3      T104         normal           1.0           15         0   

   message_length  has_exclamation  is_high_priority  
0              15                0                 1  
1              11                0                 0  
2              14                0                 1  
3              10                0                 0  


### 6.3 选择特征：能用不代表应该用

In [60]:
# 1. 选择预测时真正可用的字段
selected_features = [
    "channel",
    "customer_level",
    "has_order_id",
    "submit_hour",
    "is_night",
    "message_length",
    "has_exclamation",
]

# 2. 拆分特征 X 和标签 y
X_demo = raw_tickets[selected_features]
y_demo = raw_tickets["is_high_priority"]

# 3. 输出最终准备交给模型的内容
print("特征 X：")
print(X_demo)
print("标签 y：")
print(y_demo)

特征 X：
  channel customer_level  has_order_id  submit_hour  is_night  message_length  \
0   phone            vip           1.0           23         1              15   
1     web         normal           NaN           10         0              11   
2     app            NaN           1.0            2         1              14   
3   email         normal           1.0           15         0              10   

   has_exclamation  
0                0  
1                0  
2                0  
3                0  
标签 y：
0    1
1    0
2    1
3    0
Name: is_high_priority, dtype: int64


### 6.4 处理缺失值

`SimpleImputer` 可以根据已有数据学习填充值，再完成缺失值填充：

| 特征类型 | 示例                | 常见填充方式        |
| ---- | ----------------- | ------------- |
| 数值特征 | 是否提供订单号、提交小时、消息长度 | 使用中位数填充       |
| 类别特征 | 提交渠道、客户等级         | 使用出现次数最多的类别填充 |

In [61]:
import pandas as pd
from sklearn.impute import SimpleImputer

# 后续处理需要使用的数值特征和类别特征
#数值特征
numeric_features = [
    'has_order_id',
    'submit_hour',
    'is_night',
    'message_length',
    'has_exclamation',
]
#类别特征
category_features = ['channel', 'customer_level']

In [62]:
# -------------------- 数值特征缺失值 --------------------

# 取出包含缺失值的数值特征
numeric_demo = X_demo[numeric_features]

#创建填充器
# strategy="median"：使用每一列的中位数填充该列的缺失值
numeric_imputer = SimpleImputer(
    strategy='median',
)

# fit_transform():计算并填充
# fit: 先计算各列的中位数， transform：再完成缺失值填充
numeric_filled = numeric_imputer.fit_transform(numeric_demo)

# 转换回 DataFrame，便于查看结果
numeric_filled = pd.DataFrame(
    numeric_filled,
    columns=numeric_features,
)

print("数值特征填充结果：")
print(numeric_filled)

数值特征填充结果：
   has_order_id  submit_hour  is_night  message_length  has_exclamation
0           1.0         23.0       1.0            15.0              0.0
1           1.0         10.0       0.0            11.0              0.0
2           1.0          2.0       1.0            14.0              0.0
3           1.0         15.0       0.0            10.0              0.0


In [63]:
# -------------------- 类别特征缺失值 --------------------

# 取出包含缺失值的类别特征
category_demo = X_demo[category_features]

category_imputer = SimpleImputer(
    # 使用每一列中出现次数最多的类别填充该列的缺失值
    strategy='most_frequent',
)

# 先找出各列出现次数最多的类别，再完成缺失值填充
category_filled = category_imputer.fit_transform(category_demo)

#转换回 DataFrame，便于查看结果
category_filled = pd.DataFrame(
    category_filled,
    columns=category_features,
)

print("类别特征填充结果：")
print(category_filled)

类别特征填充结果：
  channel customer_level
0   phone            vip
1     web         normal
2     app         normal
3   email         normal


- `strategy="median"` 表示使用每一列的中位数填充；
- `strategy="most_frequent"` 表示使用每一列中出现次数最多的值填充；
- 不同特征应根据数据含义选择合适的填充方式，不能一律填成 `0`；
- 正式训练时，填充值只能根据训练集计算，不能从验证集或测试集中学习。

### 6.5 One-Hot 编码：将类别特征转换成数值(0/1)

| 原始渠道    | `channel_app` | `channel_email` | `channel_phone` | `channel_web` |
| ------- | ------------: | --------------: | --------------: | ------------: |
| `phone` |             0 |               0 |               1 |             0 |
| `web`   |             0 |               0 |               0 |             1 |
| `app`   |             1 |               0 |               0 |             0 |
| `email` |             0 |               1 |               0 |             0 |

下面使用 `OneHotEncoder` 转换类别特征：

In [64]:
import pandas as pd
from sklearn.preprocessing import OneHotEncoder

In [65]:
# 1. 使用已经填充缺失值的类别特征
category_demo = category_filled.copy()

# 2. 创建 One-Hot 编码器（生成0/1，1代表是）
category_encoder = OneHotEncoder(
    #遇到训练时没有出现过的新类别时不报错，对应编码全部设为 0
    handle_unknown='ignore',
    # 返回普通二维数组，方便转换成 DataFrame 查看
    sparse_output=False,
)

# 3. 学习已有类别并完成编码
category_encoded = category_encoder.fit_transform(category_demo)

# 4. 获取编码后生成新的特征名称
cencoded_feature_names = category_encoder.get_feature_names_out(
    category_features
)

# 5. 转换回 DataFrame，便于查看结果
category_encoded = pd.DataFrame(
    category_encoded,
    columns=cencoded_feature_names,
)

print("One-Hot 编码结果：")
print(category_encoded)

One-Hot 编码结果：
   channel_app  channel_email  channel_phone  channel_web  \
0          0.0            0.0            1.0          0.0   
1          0.0            0.0            0.0          1.0   
2          1.0            0.0            0.0          0.0   
3          0.0            1.0            0.0          0.0   

   customer_level_normal  customer_level_vip  
0                    0.0                 1.0  
1                    1.0                 0.0  
2                    1.0                 0.0  
3                    1.0                 0.0  


### 6.6 标准化：调整数值特征的尺度

`StandardScaler` 会根据每一列的“平均值”和“标准差”进行转换：

In [66]:
import pandas as pd
from sklearn.preprocessing import StandardScaler

In [67]:
# 1. 使用已经填充缺失值的数值特征
numeric_demo = numeric_filled.copy()

# 2. 创建标准化工具
numeric_scaler = StandardScaler()

# 3. 学习每一列的均值和标准差，并完成标准化
numeric_scaled = numeric_scaler.fit_transform(numeric_demo)

# 4. 转换回 DataFrame，便于查看结果
numeric_scaled = pd.DataFrame(
    numeric_scaled,
    columns=numeric_features,
)

print("标准化结果：")
print(numeric_scaled.round(3))

标准化结果：
   has_order_id  submit_hour  is_night  message_length  has_exclamation
0           0.0        1.376       1.0           1.213              0.0
1           0.0       -0.328      -1.0          -0.728              0.0
2           0.0       -1.376       1.0           0.728              0.0
3           0.0        0.328      -1.0          -1.213              0.0


标准化后需要注意：

- 数值可能出现小数或负数，这不代表数据错误；
- 标准化不会把所有数值都转换到 0～1；
- 标准化不会改变某个特征表达的业务含义；
- 树模型通常不依赖特征标准化；
- 正式训练时，均值和标准差只能从训练集学习。

## 七、Pipeline：把特征工程与模型连起来

### 7.1 先分清 Pipeline 与 ColumnTransformer

| 工具                  | 主要作用                 | 本节中的例子                |
| ------------------- | -------------------- | --------------------- |
| `Pipeline`          | 让多个步骤按照先后顺序执行        | 先填充再标准化；先完成全部预处理再训练模型 |
| `ColumnTransformer` | 让不同字段进入不同的转换流程，再合并结果 | 数值列进入数值流程，类别列进入类别流程   |

```plain
Pipeline：按照顺序连接步骤
ColumnTransformer：按照字段分配步骤
```

```plain
数值 Pipeline ─┐
               ├→ ColumnTransformer → 预测模型
类别 Pipeline ─┘
```

### 7.2 两种工具的基本写法

`Pipeline` 中的每个步骤由“步骤名称”和“处理对象”组成：

```plain
Pipeline([
    ("步骤名称", 处理对象),
    ("步骤名称", 处理对象),
])
```

`ColumnTransformer` 还需要指定该转换器处理哪些字段：

```plain
ColumnTransformer([
    ("处理名称", 转换器, 字段列表),
    ("处理名称", 转换器, 字段列表),
])
```

`ColumnTransformer` 默认只保留字段列表中明确指定的列，其他列不会自动进入模型

### 7.3 建立完整的处理与预测流程

In [68]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [69]:
"""
Pipeline([
    ("步骤名称", 处理对象),
    ("步骤名称", 处理对象),
])
"""
# 1. 数值特征依次完成缺失值填充和标准化
numeric_pipeline = Pipeline([
    (
        # "imputer" 是当前步骤的名称(转换器:处理缺失值)
        "imputer",
        # 使用每个数值特征的中位数填充该列缺失值
        SimpleImputer(strategy="median"),
    ),
    (
        # "scaler" 是当前步骤的名称(缩放器)
        "scaler",
        # 将不同数值特征调整到相近的尺度
        StandardScaler(),
    ),
])

In [70]:
# 2. 类别特征依次完成缺失值填充和 One-Hot 编码
category_pipeline = Pipeline([
    (
        # "imputer" 是当前步骤的名称
        "imputer",
        # 默认识别 NaN，并使用每列出现次数最多的类别填充
        SimpleImputer(strategy="most_frequent"),
    ),
    (
        # "onehot" 是当前步骤的名称(转换成0/1, 1表示是)
        "onehot",
        # 将字符串类别编码成数值；遇到新类别时不报错
        OneHotEncoder(handle_unknown="ignore"),
    ),
])

In [71]:
"""
ColumnTransformer([
    ("处理名称", 转换器, 字段列表),
    ("处理名称", 转换器, 字段列表),
])
"""
# 3. 让数值列和类别列分别进入对应的处理流程
preprocessor = ColumnTransformer([
    # numeric_features 中的字段进入数值处理流程
    ("numeric", numeric_pipeline, numeric_features),
    # category_features 中的字段进入类别处理流程
    ("category", category_pipeline, category_features),
])

In [72]:
# 4. 把全部特征处理步骤和逻辑回归模型连接起来
ticket_pipeline = Pipeline([
    # 先完成缺失值填充、标准化(数值特征)和 One-Hot 编码(类别特征)
    ("preprocessor", preprocessor),
    (
        # 再使用处理后的全部特征训练分类模型
        "classifier",
        # 最多迭代 1000 次，避免模型在尚未收敛时停止
        LogisticRegression(max_iter=1000),
    ),
])

# 5. 依次完成预处理规则学习和逻辑回归模型训练
# X_demo 是输入特征，y_demo 是是否为高优先级的分类标签
ticket_pipeline.fit(X_demo, y_demo)

# 6. 准备一条结构与 X_demo 相同的新工单
new_ticket_pipeline = pd.DataFrame(
    [
        {
            "channel": "wechat",
            "customer_level": "vip",
            "has_order_id": 1,
            "submit_hour": 23,
            "is_night": 1,
            "message_length": 18,
            "has_exclamation": 1,
        }
    ]
)

# 7. Pipeline 会自动处理新工单，再输出预测类别
predicted_class = ticket_pipeline.predict(new_ticket_pipeline)[0]

# predict_proba() 输出各类别的概率
# [0, 1] 表示第 1 条工单属于类别 1（高优先级）的概率
predicted_probability = ticket_pipeline.predict_proba(
    new_ticket_pipeline
)[0, 1]

print("预测类别：", predicted_class)
print(f"高优先级概率：{predicted_probability:.3f}")

预测类别： 1
高优先级概率：0.929


```plain
# 训练阶段
ticket_pipeline.fit(X_train, y_train)

# 验证、测试或预测阶段
ticket_pipeline.predict(X_new)
ticket_pipeline.predict_proba(X_new)
```

Pipeline 可以统一训练和预测时的数据处理规则，降低数据泄漏风险，但不能代替数据集划分。仍然必须先划分数据，并且只在训练集上调用 `fit`：

```plain
先划分训练集、验证集和测试集
→ ticket_pipeline.fit(X_train, y_train)
→ ticket_pipeline.predict(X_valid)
→ 方案确定后再预测 X_test
```

## 九、当日实战：工单优先级分类基线

In [73]:
from pathlib import Path

import pandas as pd

# 1. 读取课堂数据
# Path 用于表示文件路径，便于在不同操作系统中使用
data_path = Path("data/02_tickets.csv")
# read_csv() 将 CSV 文件读取为 DataFrame
tickets = pd.read_csv(data_path)

# 2. 查看数据规模、字段类型和前 5 行
# head() 默认显示前 5 行，用于快速观察数据内容
print(tickets.head())
# shape 返回（行数, 列数）
print("数据形状：", tickets.shape)
# info() 显示字段名称、非空数量和数据类型
tickets.info()

# 3. 检查缺失、重复和标签分布
print("缺失值：")
# isna().sum() 统计每一列的缺失值数量
print(tickets.isna().sum())
# duplicated().sum() 统计完全重复的行数
print("重复行数量：", tickets.duplicated().sum())
print("标签分布：")
# value_counts() 统计普通工单和高优先级工单各有多少条
print(tickets["is_high_priority"].value_counts())

  ticket_id channel customer_level  has_order_id  submit_hour  message_length  \
0      T001     web         normal             1            9              42   
1      T002     app            vip             1           23             180   
2      T003   phone            vip             1           21             156   
3      T004     web         normal             0           14              35   
4      T005   email         normal             1           10              68   

   is_high_priority  
0                 0  
1                 1  
2                 1  
3                 0  
4                 0  
数据形状： (60, 7)
<class 'pandas.DataFrame'>
RangeIndex: 60 entries, 0 to 59
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   ticket_id         60 non-null     str  
 1   channel           60 non-null     str  
 2   customer_level    60 non-null     str  
 3   has_order_id      60 non-null     int64
 4   s

`ticket_id` 只用于标识和追踪工单，不作为本次模型特征。编号通常不包含稳定的业务规律，把它交给模型可能产生没有意义的关联。

In [74]:
# 使用断言检查关键业务约束
# 工单编号不能重复
assert tickets["ticket_id"].is_unique, "ticket_id 应保持唯一"
# 提交小时必须在 0～23
assert tickets["submit_hour"].between(0, 23).all(), "提交小时应在 0～23"
# has_order_id 只能使用 0 或 1
assert tickets["has_order_id"].isin([0, 1]).all(), "是否有订单号只能是 0 或 1"
# 分类标签只能使用 0 或 1
assert tickets["is_high_priority"].isin([0, 1]).all(), "标签只能是 0 或 1"

print("基础数据质量检查通过")

基础数据质量检查通过


In [75]:
from sklearn.model_selection import train_test_split

# 1. 记录长描述特征使用的固定业务阈值
LONG_MESSAGE_THRESHOLD = 120

# 2. 定义可复用的特征构造函数
def add_ticket_features(ticket_frame: pd.DataFrame) -> pd.DataFrame:
    """使用同一套规则构造训练和预测所需的业务特征。"""
    result = ticket_frame.copy()

    # 根据提交小时判断是否属于深夜时段
    result['is_night'] = (
        (result['submit_hour'] >= 22)
        | (result['submit_hour'] < 6)
    ).astype(int)

    # 根据文本长度判断是否属于长描述
    result['is_long_message'] = (
        (result['message_length'] >= LONG_MESSAGE_THRESHOLD)
    ).astype(int)
    return result

`is_night` 和 `is_long_message` 只使用当前工单自身的字段和提前确定的固定规则，不需要从全部数据中学习统计量，因此可以在划分前构造。它们是否真正有用，仍然需要通过验证结果判断。

In [76]:
# 3. 为全部原始工单构造业务特征
tickets = add_ticket_features(tickets)

# 4. 指定模型输入列
feature_columns = [
    "channel",
    "customer_level",
    "has_order_id",
    "submit_hour",
    "message_length",
    "is_night",
    "is_long_message",
]

In [77]:
# 5. 拆分特征 X 和标签 y
X_demo = tickets[feature_columns]
y_demo = tickets["is_high_priority"]

# 6. 先划分训练集与临时数据集
X_train, X_temp, y_train, y_temp = train_test_split(
    X_demo,
    y_demo,
    # 40% 进入临时数据集，剩余 60% 作为训练集
    test_size=0.4,
    # 固定随机种子，使每次划分结果一致
    random_state=42,
    # 尽量保持训练集和临时数据集的标签比例一致
    stratify=y_demo,
)

# 7. 再把临时数据平分为验证集和测试集
X_valid, X_test, y_valid, y_test = train_test_split(
    X_temp,
    y_temp,
    # 临时数据的一半作为测试集，即全部数据的 20%
    test_size=0.5,
    # 固定随机种子，使每次划分结果一致
    random_state=42,
    # 尽量保持验证集和测试集的标签比例一致
    stratify=y_temp,
)

In [78]:
# 8. 检查三个数据集的规模和标签比例
print("训练集：", X_train.shape)
print("验证集：", X_valid.shape)
print("测试集：", X_test.shape)

# normalize=True 返回比例，sort_index() 按标签 0、1 排序
print("训练集标签比例：")
print(y_train.value_counts(normalize=True).sort_index())
print("验证集标签比例：")
print(y_valid.value_counts(normalize=True).sort_index())
print("测试集标签比例：")
print(y_test.value_counts(normalize=True).sort_index())

训练集： (36, 7)
验证集： (12, 7)
测试集： (12, 7)
训练集标签比例：
is_high_priority
0    0.527778
1    0.472222
Name: proportion, dtype: float64
验证集标签比例：
is_high_priority
0    0.5
1    0.5
Name: proportion, dtype: float64
测试集标签比例：
is_high_priority
0    0.5
1    0.5
Name: proportion, dtype: float64


缺失值填充、标准化和类别编码需要从数据中学习规则，所以只能在划分后通过 Pipeline 在训练集上拟合。`random_state=42` 用于复现数据划分，`stratify` 用于让三个集合的标签比例尽量接近。

### 9.3 建立多数类参照

In [79]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score

# 1. 创建“永远预测训练集中多数类别”的参照方案
dummy_model = DummyClassifier(
    # 忽略输入特征，始终预测训练标签中数量最多的类别
    strategy="most_frequent",
)

# 2. 从训练标签中找出数量最多的类别
dummy_model.fit(X_train, y_train)

# 3. 对验证集中的每条工单输出多数类别
dummy_predictions = dummy_model.predict(X_valid)

# 4. 比较真实标签与预测标签，计算预测正确的比例
dummy_accuracy = accuracy_score(y_valid, dummy_predictions)

print(f"多数类参照 Accuracy：{dummy_accuracy:.3f}")

多数类参照 Accuracy：0.500


`DummyClassifier` 会忽略工单特征，它不是实际业务模型，只用于提供最低比较起点。

### 9.4 训练第一版逻辑回归模型

In [80]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# 1. 按数据类型划分字段
#数值特征
numeric_features = [
    "has_order_id",
    "submit_hour",
    "message_length",
    "is_night",
    "is_long_message",
]
#类别特征
category_features = ["channel", "customer_level"]

# 2. 数值列：使用中位数填充，再进行标准化
numeric_pipeline = Pipeline([
    (
        # "imputer" 是缺失值填充步骤的名称
        "imputer",
        # 使用每个数值特征的中位数填充该列缺失值
        SimpleImputer(strategy="median"),
    ),
    (
        # "scaler" 是标准化步骤的名称
        "scaler",
        # 将不同数值特征调整到相近的尺度
        StandardScaler(),
    ),
])

# 3. 类别列：使用众数填充，再进行 One-Hot 编码
category_pipeline = Pipeline([
    (
        # "imputer" 是缺失值填充步骤的名称
        "imputer",
        # 使用每个类别特征中出现次数最多的类别填充缺失值
        SimpleImputer(strategy="most_frequent"),
    ),
    (
        # "onehot" 是类别编码步骤的名称
        "onehot",
        # 遇到训练时没有出现过的新类别时不报错
        OneHotEncoder(handle_unknown="ignore"),
    ),
])

# 4. 让不同字段进入各自的预处理流程
preprocessor = ColumnTransformer([
    # numeric_features 中的字段进入数值处理流程
    ("numeric", numeric_pipeline, numeric_features),
    # category_features 中的字段进入类别处理流程
    ("category", category_pipeline, category_features),
])

# 5. 将全部预处理步骤和逻辑回归模型串联起来
ticket_model = Pipeline([
    # 先完成数值特征和类别特征的预处理
    ("preprocessor", preprocessor),
    (
        # 再使用处理后的全部特征训练分类模型
        "classifier",
        # 最多迭代 1000 次，避免模型在尚未收敛时停止
        LogisticRegression(max_iter=1000),
    ),
])

# 6. 使用训练集同时拟合预处理规则和逻辑回归模型
# Pipeline 会先处理 X_train，再使用处理后的特征和 y_train 训练模型
ticket_model.fit(X_train, y_train)

# 7. 在验证集上进行预测
# Pipeline 会自动使用训练阶段学到的预处理规则转换 X_valid
valid_predictions = ticket_model.predict(X_valid)

# 8. 计算模型 Accuracy，并与多数类参照比较
# accuracy_score() 计算验证集中预测正确的工单比例
valid_accuracy = accuracy_score(y_valid, valid_predictions)
print(f"逻辑回归验证集 Accuracy：{valid_accuracy:.3f}")
print(
    "相对多数类参照提升："
    f"{valid_accuracy - dummy_accuracy:.3f}"
)


逻辑回归验证集 Accuracy：0.833
相对多数类参照提升：0.333


### 9.5 预测一条新工单

In [81]:
# 1. 准备一条没有标签的新工单
new_ticket = pd.DataFrame([{
    "channel": "phone",
    "customer_level": "vip",
    "has_order_id": 1,
    "submit_hour": 23,
    "message_length": 180,
}])

In [82]:
# 2. 使用与训练阶段相同的函数构造业务特征
new_ticket = add_ticket_features(new_ticket)

In [83]:
# 3. 为全部原始工单构造业务特征
tickets = add_ticket_features(tickets)

# 4. 指定模型输入列
feature_columns = [
    "channel",
    "customer_level",
    "has_order_id",
    "submit_hour",
    "message_length",
    "is_night",
    "is_long_message",
]


In [84]:
# 3. 输出预测类别和高优先级概率
# predict() 输出类别，[0] 取出第 1 条工单的预测结果
prediction = ticket_model.predict(new_ticket)[0]

# predict_proba() 输出各类别概率
# [0, 1] 表示第 1 条工单属于类别 1（高优先级）的概率
probability = ticket_model.predict_proba(new_ticket)[0, 1]

In [85]:
# 根据预测类别输出便于业务人员理解的文字
print("预测结果：", "高优先级" if prediction == 1 else "普通")
print(f"高优先级概率：{probability:.3f}")

预测结果： 高优先级
高优先级概率：0.997


### 9.6 保存实验记录

In [86]:
import json
from pathlib import Path

# 1. 汇总实验所需的关键配置和结果
experiment_record = {
    # 实验的基本信息
    "experiment_name": "02_ticket_priority_baseline",
    "data_file": str(data_path),
    "sample_count": int(len(tickets)),
    # 本次实验使用的特征、标签和特征构造规则
    "feature_columns": feature_columns,
    "target": "is_high_priority",
    "long_message_threshold": LONG_MESSAGE_THRESHOLD,
    # 数据划分比例和随机种子
    "split_ratio": {
        "train": 0.6,
        "validation": 0.2,
        "test": 0.2,
    },
    "random_state": 42,
    # 评价指标、参照方案、候选模型和验证结果
    "validation_metric": "accuracy",
    "reference_baseline": "DummyClassifier(strategy='most_frequent')",
    "candidate_model": "LogisticRegression(max_iter=1000)",
    "dummy_valid_accuracy": round(float(dummy_accuracy), 3),
    "model_valid_accuracy": round(float(valid_accuracy), 3),
}

# 2. 创建输出目录
output_dir = Path("outputs")
# exist_ok=True 表示目录已经存在时不报错
output_dir.mkdir(exist_ok=True)

# 3. 将实验记录保存为格式化 JSON 文件
record_path = output_dir / "02_experiment.json"
record_path.write_text(
    # ensure_ascii=False 保留中文，indent=2 让 JSON 缩进显示
    json.dumps(experiment_record, ensure_ascii=False, indent=2),
    # 使用 UTF-8 编码保存中文
    encoding="utf-8",
)

print(f"实验记录已保存：{record_path}")

实验记录已保存：outputs\02_experiment.json
